# 08 — ECOD on Train + Val (with Temporal Features)
Refit ECOD on `train ∪ val` instead of just `train`. Val is also all-normal (TARGET=0) and was previously discarded — adding it sharpens the per-feature empirical CDF in the tails, which is exactly the regime where anomalies sit.

Baseline to beat: `submission_ecod_plus_temporal.csv` at 95.28%. Writes `submission_ecod_temporal_trainval.csv`.

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import yaml
from src.features import build_features, build_temporal_features, feature_names_all
from src.models import score_ecod

cfg = yaml.safe_load((ROOT / 'configs' / 'config.yaml').read_text())
cols = feature_names_all(cfg['features']['raw'])

In [2]:
proc = ROOT / 'data' / 'processed'
raw = ROOT / cfg['paths']['raw_dir']

train = pd.read_parquet(proc / 'train.parquet')
val_raw = pd.read_csv(raw / cfg['data']['val_file'])
val_raw[cfg['features']['raw']] = val_raw[cfg['features']['raw']].fillna(0.0)
val = build_features(val_raw)

fit_df = build_temporal_features(pd.concat([train, val], ignore_index=True))
test = build_temporal_features(pd.read_parquet(proc / 'test.parquet'))
print(fit_df.shape, test.shape)

(1537961, 63) (289101, 61)


In [3]:
def mat(df):
    X = df[cols].to_numpy(dtype=np.float32, na_value=0.0)
    np.nan_to_num(X, copy=False, nan=0.0, posinf=0.0, neginf=0.0)
    return X

X_tr, X_te = mat(fit_df), mat(test)
scores = score_ecod(X_tr, X_te, cfg['models']['ecod'])
scores.min(), scores.max(), scores.mean()

[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    6.0s remaining:   18.1s


[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    9.1s finished


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    7.2s remaining:   21.7s


[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:   10.6s finished


[Parallel(n_jobs=8)]: Using backend LokyBackend with 8 concurrent workers.


[Parallel(n_jobs=8)]: Done   2 out of   8 | elapsed:    4.4s remaining:   13.2s


[Parallel(n_jobs=8)]: Done   8 out of   8 | elapsed:    7.5s finished


(np.float64(0.2404286948723483),
 np.float64(0.9999972043006542),
 np.float64(0.5982465824953269))

In [4]:
out = ROOT / 'outputs' / 'submissions'
pd.DataFrame({'ID': test['index'], 'TARGET': scores}).to_csv(out / 'submission_ecod_temporal_trainval.csv', index=False)